# Preprocessing

Produces a clean, tokenized version of every speech that can be reused for phrase detection, LDA, TF-IDF, clustering, and association mining.

All cleaning rules (stopwords, spelling normalization, tokenization) live in `src/text_processing.py` so every notebook uses exactly the same configuration.

**Outputs** (`data/processed/`):
- `ungdc_preprocessed.parquet` — one row per speech: metadata, `raw_text`, `sentences` (sentence-level token lists with stopwords kept, used for phrase detection), `tokens` / `clean_text` (stopwords removed)
- `compound_map.joblib` — learned hyphenated-compound normalization
- `compound_map_review.csv` — every compound decision with its counts, for inspection

**Preprocessing decisions**
| Step | Decision | Reason |
|---|---|---|
| Quotes / encoding | Typographic quotes → ASCII; U+FFFD → `'` | Apostrophe style changes by era (curly-only in the 2010s); VAT_80_2025 has 6 broken quotes |
| Contractions | `don't` → `do not`, `can't` → `cannot` | Negation is kept, so it should have one form |
| Possessives | `organization's` → `organization` | Avoid duplicate vocabulary |
| Hyphenated compounds | Learned from corpus (see below) | Fixes spelling variants (*co-operation*) and line-wrap breaks (*com-munity*) |
| Spelling variants | Oxford/UN spelling (-ize, -our, -re, defence, programme); *Viet Nam* → *vietnam*, *per cent* → *percent* | Variant spellings cluster in particular decades and would appear as false temporal trends |
| Sentences | Split on `. ! ? ; :` (not on newlines) | Phrases should not span sentence boundaries; many speeches are hard-wrapped |
| Stopwords | sklearn English + procedural/ceremonial terms; negations and a few content words kept | See *Stopwords* section |
| Lemmatization | None | Keeps the pipeline interpretable; revisit if vocabulary sparsity becomes a problem |

## Setup

In [1]:
from pathlib import Path
import sys
import re
import pandas as pd
import numpy as np
from collections import Counter
import joblib
from IPython.display import display

# Project paths
PROJECT_DIR = Path.cwd()

# If running from the notebooks/ directory, move up one level
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data" / "TXT"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Shared preprocessing configuration
sys.path.insert(0, str(PROJECT_DIR / "src"))
import text_processing as tp

print("Project directory:", PROJECT_DIR)
print("Input directory:", DATA_DIR)
print("Output directory:", PROCESSED_DIR)

assert DATA_DIR.exists(), f"Could not find corpus at {DATA_DIR}"

Project directory: /Users/johncreighton/CS6220/International Political Discourse Mining
Input directory: /Users/johncreighton/CS6220/International Political Discourse Mining/data/TXT
Output directory: /Users/johncreighton/CS6220/International Political Discourse Mining/data/processed


In [2]:
# Find all text files
text_files = sorted(DATA_DIR.rglob("*.txt"))

print(f"Found {len(text_files):,} text files")

# Extract country, session, and year from filenames
filename_pattern = re.compile(r"([A-Z]{2,3})_(\d+)_(\d{4})\.txt$")
folder_pattern = re.compile(r"Session\s+(\d+)\s*-\s*(\d{4})")

records = []
skipped = []

for path in text_files:
    filename_match = filename_pattern.search(path.name)
    folder_match = folder_pattern.search(path.parent.name)

    # Skip files that do not follow the expected naming convention
    if filename_match is None:
        skipped.append(str(path.relative_to(PROJECT_DIR)))
        continue

    country_code, session, year = filename_match.groups()

    records.append({
        # Store paths relative to the project so outputs are portable
        "file_path": str(path.relative_to(PROJECT_DIR)),
        "filename": path.name,
        "country_code": country_code,
        "session": int(session),
        "year": int(year),
        "folder": path.parent.name,
        "folder_session": int(folder_match.group(1)) if folder_match else None,
        "folder_year": int(folder_match.group(2)) if folder_match else None
    })

metadata = pd.DataFrame(records)

print(f"Skipped (non-speech) files: {skipped}")
print(f"Valid speech files: {len(metadata):,}")
print(f"Countries/entities: {metadata['country_code'].nunique():,}")
print(f"Year range: {metadata['year'].min()}–{metadata['year'].max()}")

display(metadata.head())

Found 11,142 text files


Skipped (non-speech) files: ['data/TXT/Session 47 - 1992/.DS_Store-to-UTF-8.txt']
Valid speech files: 11,141
Countries/entities: 200
Year range: 1946–2025


,file_path,filename,country_code,session,year,folder,folder_session,folder_year
0,data/TXT/Session 01 - 1946/ARG_01_1946.txt,ARG_01_1946.txt,ARG,1,1946,Session 01 - 1946,1,1946
1,data/TXT/Session 01 - 1946/AUS_01_1946.txt,AUS_01_1946.txt,AUS,1,1946,Session 01 - 1946,1,1946
2,data/TXT/Session 01 - 1946/BEL_01_1946.txt,BEL_01_1946.txt,BEL,1,1946,Session 01 - 1946,1,1946
3,data/TXT/Session 01 - 1946/BLR_01_1946.txt,BLR_01_1946.txt,BLR,1,1946,Session 01 - 1946,1,1946
4,data/TXT/Session 01 - 1946/BOL_01_1946.txt,BOL_01_1946.txt,BOL,1,1946,Session 01 - 1946,1,1946


## Read speech text and handle encoding

In [3]:
def read_speech(path):
    """
    Read a speech using UTF-8, falling back to CP1252
    if UTF-8 decoding fails.
    """
    path = PROJECT_DIR / path

    try:
        text = path.read_text(encoding="utf-8")
        encoding_used = "utf-8"
    except UnicodeDecodeError:
        text = path.read_text(encoding="cp1252")
        encoding_used = "cp1252"

    return text, encoding_used


texts = []
encodings = []

for path in metadata["file_path"]:
    text, encoding_used = read_speech(path)
    texts.append(text)
    encodings.append(encoding_used)

metadata["raw_text"] = texts
metadata["encoding"] = encodings

print("Encoding counts:")
display(metadata["encoding"].value_counts())

print("\nExample speech:")
print(metadata.loc[0, "raw_text"][:1500])

Encoding counts:


encoding
utf-8    11141
Name: count, dtype: int64


Example speech:
At the resumption of the first session of the General Assembly, the Argentine delegation wishes to state its views on a number of questions.
Politics are determined by circumstances. Accordingly, in making these remarks, we do not renounce the right to take decisions in the light of events; but we nevertheless think it desirable to state here certain useful principles to serve as a guide for our actions. We hope that these remarks will be interpreted at their true value. We are not sceptics, but our relative optimism would be strengthened if we found that a spirit of conciliation were gaining ground. Otherwise, we should be obliged to resume complete freedom of action, and proceed in accordance with the circumstances and the interests of the country which we represent.
We are no friends of unanimity, which is wont to disguise undesirable, if not improper pressure. Even the hottest debates in the defence of conflicting points of view are not incompatible with the desire

In [4]:
# Inspect text quality after reading
def count_replacement_chars(text):
    return text.count("\ufffd")


metadata["replacement_char_count"] = metadata["raw_text"].apply(
    count_replacement_chars
)

metadata["raw_char_count"] = metadata["raw_text"].str.len()

metadata["raw_word_count"] = metadata["raw_text"].apply(
    lambda x: len(x.split())
)

print("Replacement character summary:")
display(metadata["replacement_char_count"].describe())

print(
    "Documents with replacement characters:",
    (metadata["replacement_char_count"] > 0).sum()
)

print("\nDocuments with the most replacement characters:")
display(
    metadata.nlargest(5, "replacement_char_count")[
        ["filename", "encoding", "replacement_char_count"]
    ]
)

Replacement character summary:


count    11141.000000
mean         0.000539
std          0.056845
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          6.000000
Name: replacement_char_count, dtype: float64

Documents with replacement characters: 1

Documents with the most replacement characters:


,filename,encoding,replacement_char_count
11131,VAT_80_2025.txt,utf-8,6
0,ARG_01_1946.txt,utf-8,0
1,AUS_01_1946.txt,utf-8,0
2,BEL_01_1946.txt,utf-8,0
3,BLR_01_1946.txt,utf-8,0


## Stopwords

The stopword list is defined in `src/text_processing.py`:
- scikit-learn's English list, plus procedural (*mr, delegation, session*) and ceremonial (*congratulate, pleasure*) terms
- **Negations are kept** (*no, not, never, nor, neither*) because they change meaning
- **Some sklearn stopwords are content words here and are kept**: *third* (third world), *system* (international system), *interest* (national interest — sklearn drops *interest* but not *interests*), *un*, *serious*, *fire*
- *assembly, secretary, president* are kept because they form institutional phrases

In [5]:
print(f"Total stopwords: {len(tp.STOPWORDS):,}")
print("Negations kept:", sorted(tp.NEGATIONS))
print("Content words removed from sklearn's list:", sorted(tp.RETAINED_CONTENT_WORDS))
print("Procedural stopwords:", sorted(tp.PROCEDURAL_STOPWORDS))
print("Ceremonial stopwords:", sorted(tp.CEREMONIAL_STOPWORDS))

Total stopwords: 325
Negations kept: ['neither', 'never', 'no', 'nor', 'not']
Content words removed from sklearn's list: ['fire', 'interest', 'serious', 'system', 'third', 'un']
Procedural stopwords: ['chairman', 'chairperson', 'delegate', 'delegates', 'delegation', 'madam', 'mr', 'mrs', 'plenary', 'session']
Ceremonial stopwords: ['appreciation', 'behalf', 'congratulate', 'congratulations', 'occasion', 'pleasure', 'thanks', 'wish']


## Normalize text

### Hyphenated compounds

Many words appear both hyphenated and closed. Some are genuine spelling variants (*co-operation / cooperation*, *peace-keeping / peacekeeping*), while others are words broken across a line in the source transcripts (*com-munity*, *re-sources*, *be-tween*). The tokenizer splits on hyphens, so both cases would otherwise produce fragments.

Rather than writing these by hand, we **learn them from the corpus**: a hyphenated pair `a-b` is normalized when both it and the closed form `ab` occur (≥3 and ≥10 times). The compound is written closed when the prefix cannot stand alone (*co-, non-, re-, inter-*…), when either part is a stopword (*well-being*), or when the closed form is the majority; otherwise the rare closed form is split to match the dominant spelling (*secretarygeneral* → *secretary general*). A short exclusion list prevents false merges such as *vis-a* → *visa*.

In [6]:
compound_map, compound_review = tp.learn_compound_map(metadata["raw_text"])
compound_review = pd.DataFrame(compound_review).sort_values(
    "hyphenated_count", ascending=False
)

print("Compound decisions:")
display(compound_review["direction"].value_counts())

print("Most frequent compounds joined into one word:")
display(compound_review[compound_review["direction"] == "join"].head(15))

print("Compounds where the rare closed form is split:")
display(compound_review[compound_review["direction"] == "split"].head(10))

print("Examples of repaired line-wrap breaks (closed form far more common):")
display(
    compound_review[
        (compound_review["direction"] == "join")
        & (compound_review["closed_count"] > 50 * compound_review["hyphenated_count"])
    ].head(10)
)

compound_review.to_csv(PROCESSED_DIR / "compound_map_review.csv", index=False)

Compound decisions:


direction
join        495
split        36
excluded      2
Name: count, dtype: int64

Most frequent compounds joined into one word:


,hyphenated,closed,hyphenated_count,closed_count,direction
4,co-operation,cooperation,20173,22973,join
9,self-determination,selfdetermination,8404,15,join
242,non-aligned,nonaligned,5394,495,join
239,peace-keeping,peacekeeping,4038,4256,join
21,so-called,socalled,4007,15,join
284,non-proliferation,nonproliferation,3838,117,join
15,well-being,wellbeing,3761,269,join
234,super-powers,superpowers,2117,434,join
1,co-operate,cooperate,1975,1760,join
66,under-developed,underdeveloped,1635,707,join


Compounds where the rare closed form is split:


,hyphenated,closed,hyphenated_count,closed_count,direction
3,secretary-general,secretarygeneral,23658,254,split
78,viet-nam,vietnam,4372,288,split
62,south-east,southeast,2727,609,split
25,long-term,longterm,2637,76,split
50,cease-fire,ceasefire,1542,1067,split
260,north-south,northsouth,1389,132,split
309,guinea-bissau,guineabissau,1258,30,split
98,east-west,eastwest,1026,44,split
76,long-standing,longstanding,971,211,split
265,sea-bed,seabed,921,343,split


Examples of repaired line-wrap breaks (closed form far more common):


,hyphenated,closed,hyphenated_count,closed_count,direction
153,inter-national,international,269,156252,join
235,inter-dependence,interdependence,27,2769,join
389,com-munity,community,27,46352,join
215,re-sources,resources,25,20335,join
339,under-standing,understanding,24,8434,join
345,on-going,ongoing,23,3056,join
149,dis-armament,disarmament,18,21915,join
297,multi-lateral,multilateral,18,7755,join
71,inter-dependent,interdependent,17,1117,join
187,pro-gramme,programme,17,9821,join


In [7]:
# Apply normalization (quotes, contractions, possessives, compounds, spelling)
metadata["norm_text"] = [
    tp.normalize_text(text, compound_map) for text in metadata["raw_text"]
]

# Evidence that the variant spellings were era-specific and are now unified
variant_patterns = {
    "co-operat (raw)":   (r"\bco-\s*operat", "raw_text"),
    "cooperat (raw)":    (r"\bcooperat", "raw_text"),
    "cooperat (normalized)": (r"\bcooperat", "norm_text"),
    "program (raw)":     (r"\bprograms?\b", "raw_text"),
    "programme (normalized)": (r"\bprogramme", "norm_text"),
    "center (raw)":      (r"\bcenters?\b", "raw_text"),
    "viet nam (raw)":    (r"\bviet[\s-]+nam\b", "raw_text"),
    "vietnam (normalized)": (r"\bvietnam\b", "norm_text"),
    "curly apostrophe (raw)": ("\u2019", "raw_text"),
}

decade = metadata["year"] // 10 * 10
variant_table = pd.DataFrame({
    label: metadata[column].str.lower().str.count(pattern).groupby(decade).sum()
    for label, (pattern, column) in variant_patterns.items()
}).T

print("Occurrences by decade")
display(variant_table)

Occurrences by decade


year,1940,1950,1960,1970,1980,1990,2000,2010,2020
co-operat (raw),483,2256,4466,7329,7460,1145,10,4,27
cooperat (raw),43,273,419,1110,2101,7980,5696,5023,3319
cooperat (normalized),518,2525,4878,8429,9553,9124,5706,5027,3345
program (raw),0,6,3,990,1011,67,11,9,209
programme (normalized),79,891,1592,1975,3091,3741,2664,2359,903
center (raw),0,1,3,233,200,17,38,26,125
viet nam (raw),8,157,3523,1786,931,179,146,169,78
vietnam (normalized),8,161,3541,1997,949,190,148,170,102
curly apostrophe (raw),741,3085,3584,63,194,7955,8304,14605,7996


The raw counts show that variant spellings are concentrated in particular decades (e.g. *co-operation* dominates before 1990 and almost disappears afterwards; *program*, *center* appear mostly in the 1970s–80s; curly apostrophes appear only in some periods). These reflect transcription conventions, not changes in political language, so without normalization they would surface as spurious temporal trends in TF-IDF and LDA.

## Clean and Tokenize the Speeches

In [8]:
# Sentence-level tokens with stopwords preserved (input to phrase detection)
metadata["sentences"] = metadata["norm_text"].apply(tp.split_sentences)

# Flat token list with stopwords removed (for general analysis)
metadata["tokens"] = metadata["sentences"].apply(
    lambda sentences: tp.remove_stopwords(
        [token for sentence in sentences for token in sentence]
    )
)

metadata["clean_text"] = metadata["tokens"].apply(" ".join)
metadata["clean_word_count"] = metadata["tokens"].apply(len)
metadata["sentence_count"] = metadata["sentences"].apply(len)

print(f"Total sentences: {metadata['sentence_count'].sum():,}")

display(
    metadata[
        ["filename", "raw_word_count", "sentence_count", "clean_word_count"]
    ].head(10)
)

Total sentences: 1,258,550


,filename,raw_word_count,sentence_count,clean_word_count
0,ARG_01_1946.txt,3364,150,1459
1,AUS_01_1946.txt,4531,132,2039
2,BEL_01_1946.txt,2501,105,1109
3,BLR_01_1946.txt,3055,107,1453
4,BOL_01_1946.txt,1501,62,614
5,BRA_01_1946.txt,1488,60,631
6,CAN_01_1946.txt,1521,55,668
7,CHL_01_1946.txt,1376,49,639
8,CHN_01_1946.txt,2946,102,1346
9,COL_01_1946.txt,3861,121,1694


In [9]:
# Compare before and after preprocessing for a randomly selected speech
sample = metadata.sample(1, random_state=42).iloc[0]

print("FILE:", sample["filename"])

print("\n--- ORIGINAL TEXT ---\n")
print(sample["raw_text"][:1200])

print("\n--- CLEANED TEXT ---\n")
print(sample["clean_text"][:1200])

print("\n--- FIRST 50 TOKENS ---\n")
print(sample["tokens"][:50])

FILE: QAT_54_1999.txt

--- ORIGINAL TEXT ---

I extend
my congratulations to Mr. Theo-Ben Gurirab, on his
election as President of the General Assembly at its fifty-
fourth session. I wish him success in guiding the work of
the Assembly.
It gives me great pleasure to extend my thanks to his
predecessor, Mr. Didier Opertti, for his effective work as
President of the Assembly at its previous session. I wish
him similar success in his future endeavours.
Allow me on this occasion to congratulate the three
new Members on their admission to the United Nations.
4


Allow me also, on behalf of my Government, to express
appreciation to the Secretary-General, for his tireless work
and laudable efforts in the midst of an international
situation that can be described — at best — as difficult and
complicated.
Of course the international situation is truly difficult
and complicated. It is no coincidence that as we meet in
this forum to discuss the affairs and state of the world
today — at this, the 

In [10]:
# Vocabulary after preprocessing
word_counts = Counter(
    token
    for document_tokens in metadata["tokens"]
    for token in document_tokens
)

vocabulary = pd.DataFrame(
    word_counts.most_common(50),
    columns=["word", "frequency"]
)

print(f"Total tokens: {sum(word_counts.values()):,}")
print(f"Unique vocabulary terms: {len(word_counts):,}")

display(vocabulary)

Total tokens: 15,436,042
Unique vocabulary terms: 61,985


,word,frequency
0,nations,188570
1,united,186115
2,international,156524
3,world,133377
4,not,128309
5,countries,118897
6,peace,109142
7,states,91503
8,people,89693
9,development,87096


In [11]:
# Preprocessing statistics
metadata["token_retention_pct"] = (
    metadata["clean_word_count"] /
    metadata["raw_word_count"].replace(0, np.nan)
) * 100

print("Preprocessing summary:")
display(
    metadata[
        [
            "raw_word_count",
            "clean_word_count",
            "token_retention_pct"
        ]
    ].describe()
)

print("\nDocuments with no remaining tokens:")
print((metadata["clean_word_count"] == 0).sum())

print("\nShortest cleaned speeches:")
display(
    metadata.nsmallest(10, "clean_word_count")[
        ["filename", "year", "country_code", "clean_word_count"]
    ]
)

Preprocessing summary:


,raw_word_count,clean_word_count,token_retention_pct
count,11141.000000,11141.000000,11141.000000
mean,2901.259582,1385.516740,48.284881
std,1496.559264,673.948886,2.834381
min,423.000000,170.000000,35.138249
25%,1865.000000,919.000000,46.385921
50%,2550.000000,1242.000000,48.230594
75%,3622.000000,1713.000000,50.166223
max,22003.000000,9543.000000,60.669975



Documents with no remaining tokens:
0

Shortest cleaned speeches:


,filename,year,country_code,clean_word_count
2044,EGY_28_1973.txt,1973,EGY,170
3960,VCT_41_1986.txt,1986,VCT,200
19,IRN_01_1946.txt,1946,IRN,219
73,URY_02_1947.txt,1947,URY,219
17,HTI_01_1946.txt,1946,HTI,221
9219,BEN_71_2016.txt,2016,BEN,226
4467,DDR_45_1990.txt,1990,DDR,237
9700,LTU_73_2018.txt,2018,LTU,248
10246,GNB_76_2021.txt,2021,GNB,251
10327,RWA_76_2021.txt,2021,RWA,253


## Save processed corpus

In [12]:
output_path = PROCESSED_DIR / "ungdc_preprocessed.parquet"

columns_to_save = [
    "filename",
    "country_code",
    "session",
    "year",
    "file_path",
    "encoding",
    "raw_word_count",
    "clean_word_count",
    "sentence_count",
    "replacement_char_count",
    "raw_text",
    "sentences",
    "clean_text",
    "tokens"
]

processed = metadata[columns_to_save].copy()

processed.to_parquet(
    output_path,
    index=False,
    engine="pyarrow"
)

joblib.dump(compound_map, PROCESSED_DIR / "compound_map.joblib")

print(f"Saved {len(processed):,} speeches to:")
print(output_path)

Saved 11,141 speeches to:
/Users/johncreighton/CS6220/International Political Discourse Mining/data/processed/ungdc_preprocessed.parquet


In [13]:
# Validate
# Read the saved dataset back in
check = pd.read_parquet(output_path)

print(f"Saved documents: {len(check):,}")
print(f"Unique countries/entities: {check['country_code'].nunique():,}")
print(f"Year range: {check['year'].min()}–{check['year'].max()}")
print(f"Total cleaned tokens: {check['clean_word_count'].sum():,}")

display(check.drop(columns=["raw_text"]).head(3))

Saved documents: 11,141
Unique countries/entities: 200
Year range: 1946–2025
Total cleaned tokens: 15,436,042


,filename,country_code,session,year,file_path,encoding,raw_word_count,clean_word_count,sentence_count,replacement_char_count,sentences,clean_text,tokens
0,ARG_01_1946.txt,ARG,1,1946,data/TXT/Session 01 - 1946/ARG_01_1946.txt,utf-8,3364,1459,150,0,"[[at, the, resumption, of, the, first, session...",resumption general assembly argentine wishes s...,"[resumption, general, assembly, argentine, wis..."
1,AUS_01_1946.txt,AUS,1,1946,data/TXT/Session 01 - 1946/AUS_01_1946.txt,utf-8,4531,2039,132,0,"[[the, general, assembly, of, the, united, nat...",general assembly united nations meeting time h...,"[general, assembly, united, nations, meeting, ..."
2,BEL_01_1946.txt,BEL,1,1946,data/TXT/Session 01 - 1946/BEL_01_1946.txt,utf-8,2501,1109,105,0,"[[the, principal, organs, of, the, united, nat...",principal organs united nations functioning ne...,"[principal, organs, united, nations, functioni..."
